In [1]:
import pandas as pd
import numpy as np

# ============================================================
# 1. CITIRE
# ============================================================
# Citim normal: "NA" devine NaN. Completăm explicit, pe coloane, mai jos.
df = pd.read_csv("data/AmesHousing.csv", sep="\t")
print("Dimensiuni inițiale:", df.shape)

df_clean = df.copy()

# ============================================================
# 2. DUPLICATE ȘI SalePrice INVALID
# ============================================================
df_clean = df_clean.drop_duplicates()
df_clean = df_clean.dropna(subset=["SalePrice"])
df_clean = df_clean[df_clean["SalePrice"] > 0].copy()
print("După duplicate / SalePrice invalid:", df_clean.shape)

# ============================================================
# 3. VERIFICARE NUME DE COLOANE (fără erori tăcute)
# ============================================================
# Categorice unde lipsa = "nu există facilitatea"
absence_categories = [
    "Alley", "Bsmt Qual", "Bsmt Cond", "Bsmt Exposure",
    "BsmtFin Type 1", "BsmtFin Type 2", "Fireplace Qu",
    "Garage Type", "Garage Finish", "Garage Qual", "Garage Cond",
    "Pool QC", "Fence", "Misc Feature", "Mas Vnr Type",
]

# Numerice unde lipsa = 0 (nu există subsol / garaj / placaj)
zero_fill_numeric = [
    "Bsmt Full Bath", "Bsmt Half Bath", "BsmtFin SF 1", "BsmtFin SF 2",
    "Bsmt Unf SF", "Total Bsmt SF", "Garage Cars", "Garage Area",
    "Mas Vnr Area",
]

needed = absence_categories + zero_fill_numeric + [
    "Lot Frontage", "Neighborhood", "Electrical", "Garage Yr Blt",
    "Gr Liv Area", "Year Built", "Year Remod/Add", "Yr Sold", "MS SubClass",
]
not_found = [c for c in needed if c not in df_clean.columns]
if not_found:
    raise ValueError(f"Coloane negăsite (verifică df.columns): {not_found}")

# ============================================================
# 4. VALORI LIPSĂ
# ============================================================
# 4a. Categorice de absență -> "None"
for col in absence_categories:
    df_clean[col] = df_clean[col].fillna("None")

# 4b. Numerice de absență -> 0
df_clean[zero_fill_numeric] = df_clean[zero_fill_numeric].fillna(0)

# 4c. Lot Frontage -> mediana pe cartier (apoi mediana globală ca rezervă)
# NOTĂ: dacă faci train/test split, calculează mediana doar pe train
# (de preferat într-un Pipeline cu SimpleImputer) ca să eviți data leakage.
df_clean["Lot Frontage"] = (
    df_clean.groupby("Neighborhood")["Lot Frontage"]
            .transform(lambda s: s.fillna(s.median()))
)
df_clean["Lot Frontage"] = df_clean["Lot Frontage"].fillna(
    df_clean["Lot Frontage"].median()
)

# 4d. Electrical (1 valoare lipsă) -> moda
df_clean["Electrical"] = df_clean["Electrical"].fillna(
    df_clean["Electrical"].mode()[0]
)

# ============================================================
# 5. OUTLIERI DOCUMENTAȚI (recomandarea autorului setului)
# ============================================================
n_before = len(df_clean)
df_clean = df_clean[df_clean["Gr Liv Area"] <= 4000].copy()
print("Outlieri eliminați (Gr Liv Area > 4000):", n_before - len(df_clean))

# ============================================================
# 6. VERIFICĂRI DE CONSISTENȚĂ
# ============================================================
print("Remodelare înainte de construcție:",
      (df_clean["Year Remod/Add"] < df_clean["Year Built"]).sum())
print("Garaj construit după vânzare / an invalid:",
      ((df_clean["Garage Yr Blt"] > df_clean["Yr Sold"])).sum())
print("Vândute înainte de construcție:",
      (df_clean["Yr Sold"] < df_clean["Year Built"]).sum())

# Corecții
df_clean["Year Remod/Add"] = np.maximum(
    df_clean["Year Remod/Add"], df_clean["Year Built"]
)

# ============================================================
# 7. FEATURE-URI NOI
# ============================================================
df_clean["Has Basement"] = (df_clean["Total Bsmt SF"] > 0).astype(int)
df_clean["Has Garage"] = (df_clean["Garage Area"] > 0).astype(int)

df_clean["House Age at Sale"] = (
    (df_clean["Yr Sold"] - df_clean["Year Built"]).clip(lower=0)
)
df_clean["Years Since Remodel"] = (
    (df_clean["Yr Sold"] - df_clean["Year Remod/Add"]).clip(lower=0)
)

# Garage Yr Blt are typo-uri (ex. 2207) și e redundant cu Garage Type / Has Garage
df_clean = df_clean.drop(columns=["Garage Yr Blt"])

# ============================================================
# 7b. FEATURE ENGINEERING + ELIMINAREA REDUNDANȚELOR
# ============================================================
porch_cols = ["Open Porch SF", "Enclosed Porch", "3Ssn Porch",
              "Screen Porch", "Wood Deck SF"]
needed_fe = ["Total Bsmt SF", "Gr Liv Area", "Full Bath", "Half Bath",
             "Bsmt Full Bath", "Bsmt Half Bath", "Overall Qual",
             "Year Built", "Year Remod/Add", "Garage Cars"] + porch_cols
not_found = [c for c in needed_fe if c not in df_clean.columns]
if not_found:
    raise ValueError(f"Coloane negăsite: {not_found}")

# --- Feature-uri noi ---
df_clean["Total SF"] = df_clean["Total Bsmt SF"] + df_clean["Gr Liv Area"]
df_clean["Total Baths"] = (
    df_clean["Full Bath"] + 0.5 * df_clean["Half Bath"]
    + df_clean["Bsmt Full Bath"] + 0.5 * df_clean["Bsmt Half Bath"]
)
df_clean["Total Porch SF"] = df_clean[porch_cols].sum(axis=1)
df_clean["Qual x Area"] = df_clean["Overall Qual"] * df_clean["Gr Liv Area"]
df_clean["Is Remodeled"] = (
    df_clean["Year Remod/Add"] != df_clean["Year Built"]
).astype(int)

# --- Redundanțe evidente ---
# Year Built / Year Remod/Add sunt acoperite de House Age at Sale,
# Years Since Remodel și Is Remodeled; Garage Cars e redundant cu Garage Area.
redundant_cols = ["Year Built", "Year Remod/Add", "Garage Cars"]
df_clean = df_clean.drop(columns=redundant_cols)

# --- Coloane aproape constante (>99% aceeași valoare) ---
protected = ["SalePrice", "PID", "Order"]
near_constant = [
    c for c in df_clean.columns
    if c not in protected
    and df_clean[c].value_counts(normalize=True).iloc[0] > 0.99
]
print("Aproape constante, eliminate:", near_constant)
df_clean = df_clean.drop(columns=near_constant)

# --- Perechi numerice foarte corelate (doar informativ, nu elimină nimic) ---
corr = (df_clean.drop(columns=protected, errors="ignore")
                .select_dtypes("number").corr().abs())
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
pairs = upper.stack().loc[lambda s: s > 0.9].sort_values(ascending=False)
print("Perechi cu |corelație| > 0.9:")
print(pairs)

# ============================================================
# 8. TIPURI DE VARIABILE
# ============================================================
# MS SubClass e nominal (20, 30, 60 nu au ordine)
df_clean["MS SubClass"] = df_clean["MS SubClass"].astype(str)

# Coloane ordinale text (Ex/Gd/TA/Fa/Po) -> numere; "None" = 0
qual_map = {"None": 0, "Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5}
ordinal_qual_cols = [
    "Exter Qual", "Exter Cond", "Bsmt Qual", "Bsmt Cond", "Heating QC",
    "Kitchen Qual", "Fireplace Qu", "Garage Qual", "Garage Cond", "Pool QC",
]
# Păstrăm doar coloanele care mai există (unele pot fi eliminate la 7b)
present = [c for c in ordinal_qual_cols if c in df_clean.columns]
skipped = [c for c in ordinal_qual_cols if c not in df_clean.columns]
print("Ordinale sărite (eliminate anterior):", skipped)

for col in present:
    df_clean[col] = df_clean[col].map(qual_map)
    if df_clean[col].isna().any():
        raise ValueError(f"Valori necunoscute în {col}")

# ============================================================
# 9. (OPȚIONAL) FILTRARE CASE FĂRĂ SUBSOL
# ============================================================
# Recomandat să rămână False, dacă tema nu cere explicit filtrarea.
# Ai deja "Has Basement" care surprinde diferența.
ONLY_WITH_BASEMENT = False
if ONLY_WITH_BASEMENT:
    n_before = len(df_clean)
    df_clean = df_clean[df_clean["Total Bsmt SF"] > 0].copy()
    print("Case fără subsol eliminate:", n_before - len(df_clean))

# ============================================================
# 10. VERIFICĂRI FINALE
# ============================================================
remaining = df_clean.isna().sum().loc[lambda s: s > 0]
print("Valori lipsă rămase:", remaining.sum())
if len(remaining):
    print(remaining)
print("Duplicate rămase:", df_clean.duplicated().sum())
print("Dimensiuni finale:", df_clean.shape)

# Identificatori: îi păstrăm în fișier, dar îi excludem din predictori
id_columns = [c for c in ["PID", "Order"] if c in df_clean.columns]
print("Coloane de exclus din predictori:", id_columns)

# ============================================================
# 11. SALVARE (după toți pașii)
# ============================================================
df_clean.to_csv("data/AmesHousing_cleaned.csv", index=False)
print("Dataset salvat.")

# La modelare:
# X = df_clean.drop(columns=["SalePrice"] + id_columns)
# y = df_clean["SalePrice"]
# apoi one-hot encoding pe restul coloanelor nominale (pd.get_dummies)

Dimensiuni inițiale: (2930, 82)
După duplicate / SalePrice invalid: (2930, 82)
Outlieri eliminați (Gr Liv Area > 4000): 5
Remodelare înainte de construcție: 1
Garaj construit după vânzare / an invalid: 1
Vândute înainte de construcție: 0
Aproape constante, eliminate: ['Street', 'Utilities', 'Condition 2', 'Pool Area', 'Pool QC']
Perechi cu |corelație| > 0.9:
Gr Liv Area  Qual x Area    0.9126
dtype: float64
Ordinale sărite (eliminate anterior): ['Pool QC']
Valori lipsă rămase: 0
Duplicate rămase: 0
Dimensiuni finale: (2925, 82)
Coloane de exclus din predictori: ['PID', 'Order']
Dataset salvat.


In [9]:
pip install xgboost plotly optuna anywidget


   ---------------------------------------- 0.0/48.9 MB ? eta -:--:--
   ---------------------------------------- 0.3/48.9 MB ? eta -:--:--
   --- ------------------------------------ 4.5/48.9 MB 17.1 MB/s eta 0:00:03
   ------ --------------------------------- 7.9/48.9 MB 19.6 MB/s eta 0:00:03
   ----------- ---------------------------- 14.2/48.9 MB 20.5 MB/s eta 0:00:02
   --------------- ------------------------ 19.4/48.9 MB 21.4 MB/s eta 0:00:02
   -------------------- ------------------- 24.6/48.9 MB 22.1 MB/s eta 0:00:02
   ------------------------ --------------- 29.9/48.9 MB 22.4 MB/s eta 0:00:01
   ---------------------------- ----------- 34.9/48.9 MB 22.6 MB/s eta 0:00:01
   --------------------------------- ------ 40.6/48.9 MB 23.1 MB/s eta 0:00:01
   ------------------------------------- -- 46.1/48.9 MB 23.4 MB/s eta 0:00:01
   ---------------------------------------  48.8/48.9 MB 23.5 MB/s eta 0:00:01
   ---------------------------------------- 48.9/48.9 MB 21.5 MB/s  0:00